In [1]:
wimport numpy as np
import pandas as pd
import joblib
import torch
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, f1_score, confusion_matrix
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)

In [2]:
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)


In [3]:
# ==========================================
# CONFIG
# ==========================================
TRAIN_FILE = "TrainingDataset.csv"
VAL_FILE = "ValidationDataset.csv"
TEST_FILE = "TestingDataset.csv"
TEXT_COL = 0       # preprocessed text column
LABEL_COL = 1
MODEL_CHECKPOINT = "aubmindlab/bert-base-arabertv02"   # standard AraBERT v2 checkpoint
MAX_LENGTH = 64          # tweets are short; 64 tokens comfortably covers most
FINAL_EPOCHS = 4         # epochs for the winning config's final training run
SEARCH_EPOCHS = 3        # shorter budget per config during the grid search

In [4]:
# Note: AraBERT's own authors recommend an AraBERT-specific preprocessing step
# (farasa-based segmentation via the `arabert.preprocess.ArabertPreprocessor`
# class) for best results, separate from generic cleaning. Since preprocessing
# is already handled in your other script, this is left as an optional add-on
# rather than applied here — worth considering if AraBERT underperforms
# expectations relative to the BiLSTM baseline.

In [5]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}\n")

Using device: cuda



In [6]:
# ==========================================
# 1. LOAD DATA (already preprocessed, deduplicated, and split)
# ==========================================
train_df = pd.read_csv(TRAIN_FILE, header=None)
val_df = pd.read_csv(VAL_FILE, header=None)
test_df = pd.read_csv(TEST_FILE, header=None)

print("Dataset Sizes Loaded:")
print(f" - Train Set:      {len(train_df)} samples")
print(f" - Validation Set: {len(val_df)} samples")
print(f" - Test Set:       {len(test_df)} samples\n")


Dataset Sizes Loaded:
 - Train Set:      108786 samples
 - Validation Set: 14510 samples
 - Test Set:       21757 samples



In [7]:
train_df

,0,1
0,تشيلسي يتعادل مع مانشستر يونايتد بهدف لمثله في...,رياضه
1,فوائد لا تعرفينها من قبل عن فاكهه الدراجون,طهي
2,يدوم جمال المرأه مادام الرجل يحبها ويدوم حب ال...,خواطر واقتباسات
3,خادم الحرمين الشريفين الملك سلمان بن عبدالعزيز...,سياسه
4,بورشه تيربو الكلاسيكيه من قطع الليجو,سيارت
...,...,...
108781,وصفها ترمب بالأخبار العظيمه شركه موديرنا تعلن ...,صحه
108782,اللعبه ستكون مبنيه علي محرك ولكنها ستكون مختلف...,تكنولوجيا
108783,بطش أردوغان يتفاقم والأتراك يهربون للخارج تركيا,سياسه
108784,بلتون الماليه تحدد السهم الاستهلاكي الأكثر تأث...,اقتصاد


In [8]:
# ==========================================
# 2. LABEL ENCODING
# ==========================================
label_encoder = LabelEncoder()
train_df["label"] = label_encoder.fit_transform(train_df[LABEL_COL])
val_df["label"] = label_encoder.transform(val_df[LABEL_COL])
test_df["label"] = label_encoder.transform(test_df[LABEL_COL])

num_classes = len(label_encoder.classes_)
print(f"Classes ({num_classes}): {list(label_encoder.classes_)}\n")

Classes (10): ['اقتصاد', 'تكنولوجيا', 'خواطر واقتباسات', 'دين', 'رياضه', 'سيارت', 'سياسه', 'صحه', 'طهي', 'علوم']



In [9]:
# ==========================================
# 3. TOKENIZATION (HuggingFace Datasets + AraBERT tokenizer)
# ==========================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def tokenize_batch(batch):
    return tokenizer(
        batch[str(TEXT_COL)], # Access using string key '0'
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )

train_ds = Dataset.from_pandas(train_df[[TEXT_COL, "label"]])
val_ds = Dataset.from_pandas(val_df[[TEXT_COL, "label"]])
test_ds = Dataset.from_pandas(test_df[[TEXT_COL, "label"]])

train_ds = train_ds.map(tokenize_batch, batched=True)
val_ds = val_ds.map(tokenize_batch, batched=True)
test_ds = test_ds.map(tokenize_batch, batched=True)

keep_cols = ["input_ids", "attention_mask", "label"]
train_ds.set_format(type="torch", columns=keep_cols)
val_ds.set_format(type="torch", columns=keep_cols)
test_ds.set_format(type="torch", columns=keep_cols)

config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/381 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/825k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.64M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/datasets/table.py:728: UserWarning: The DataFrame has column names of mixed type. They will be converted to strings and not roundtrip correctly.
  return cls(pa.Table.from_pandas(*args, **kwargs))
/usr/local/lib/python3.13/dist-packages/datasets/table.py:728: UserWarning: The DataFrame has column names of mixed type. They will be converted to strings and not roundtrip correctly.
  return cls(pa.Table.from_pandas(*args, **kwargs))
/usr/local/lib/python3.13/dist-packages/datasets/table.py:728: UserWarning: The DataFrame has column names of mixed type. They will be converted to strings and not roundtrip correctly.
  return cls(pa.Table.from_pandas(*args, **kwargs))


Map:   0%|          | 0/108786 [00:00<?, ? examples/s]

Map:   0%|          | 0/14510 [00:00<?, ? examples/s]

Map:   0%|          | 0/21757 [00:00<?, ? examples/s]

In [10]:
# ==========================================
# 4. METRICS
# ==========================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }

In [11]:
# ==========================================
# 5. HYPERPARAMETER TUNING (grid search, selected on validation macro-F1)
# ==========================================
# Fine-tuning transformer models is expensive, so the grid here is kept
# deliberately small relative to the BiLSTM search — 2 learning rates x 2
# batch sizes = 4 configurations, each trained for a short SEARCH_EPOCHS
# budget. This still lets you report a tuned result rather than a single
# arbitrarily-chosen setting, consistent with how the SVM and BiLSTM
# baselines were tuned. Extend the grids below if you have GPU budget for it.
LEARNING_RATE_GRID = [2e-5, 5e-5]
BATCH_SIZE_GRID = [16, 32]

# Disable torchvision integration in datasets to prevent VideoReader import error
import datasets
datasets.config.TORCHVISION_AVAILABLE = False

best_val_f1_macro = -1.0
best_config = None
best_model_path = None

print("Starting hyperparameter grid search (selecting on validation macro-F1)...")
for lr in LEARNING_RATE_GRID:
    for batch_size in BATCH_SIZE_GRID:
        run_name = f"lr{lr}_bs{batch_size}"
        print(f"--- Config: learning_rate={lr}, batch_size={batch_size} ---")

        model = AutoModelForSequenceClassification.from_pretrained(
            MODEL_CHECKPOINT, num_labels=num_classes
        ).to(DEVICE)

        training_args = TrainingArguments(
            output_dir=f"./arabert_search_{run_name}",
            num_train_epochs=SEARCH_EPOCHS,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            learning_rate=lr,
            eval_strategy="epoch",
            save_strategy="epoch",
            save_total_limit=1,
            load_best_model_at_end=True,
            metric_for_best_model="f1_macro",
            greater_is_better=True,
            logging_steps=50,
            report_to="none",
            seed=SEED,
            optim="adamw_torch" # Changed to adamw_torch
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_ds,
            eval_dataset=val_ds,
            compute_metrics=compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
        )

        trainer.train()
        val_metrics = trainer.evaluate()
        val_f1_macro = val_metrics["eval_f1_macro"]
        print(f"    Validation Macro-F1: {val_f1_macro:.4f}\n")

        if val_f1_macro > best_val_f1_macro:
            best_val_f1_macro = val_f1_macro
            best_config = {"learning_rate": lr, "batch_size": batch_size}
            best_model_path = f"./arabert_search_{run_name}"

print("Best Hyperparameters Found:")
print(best_config)
print(f"Best Validation Macro-F1 (search budget): {best_val_f1_macro:.4f}\n")

Starting hyperparameter grid search (selecting on validation macro-F1)...
--- Config: learning_rate=2e-05, batch_size=16 ---


model.safetensors: reconstructing file:   0%|          |  0.00B /  543MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,F1 Macro,F1 Weighted
1,0.223671,0.222352,0.930659,0.931723
2,0.192252,0.224987,0.940544,0.941195
3,0.070655,0.255225,0.942122,0.942920


... (25 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,F1 Macro,F1 Weighted
0.051073,0.229015,3,0.944179,0.944906


    Validation Macro-F1: 0.9442

Best Hyperparameters Found:
{'learning_rate': 5e-05, 'batch_size': 32}
Best Validation Macro-F1 (search budget): 0.9442



In [ ]:
# ==========================================
# 6. FINAL TRAINING RUN (winning config, fuller epoch budget)
# ==========================================
final_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT, num_labels=num_classes
).to(DEVICE)

final_training_args = TrainingArguments(
    output_dir="./arabert_final",
    num_train_epochs=FINAL_EPOCHS,
    per_device_train_batch_size=best_config["batch_size"],
    per_device_eval_batch_size=best_config["batch_size"],
    learning_rate=best_config["learning_rate"],
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=50,
    report_to="none",
    seed=SEED,
    optim="adamw_torch" # Changed to adamw_torch
)

final_trainer = Trainer(
    model=final_model,
    args=final_training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print("Training final model with best hyperparameters...")
final_trainer.train()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Training final model with best hyperparameters...


Epoch,Training Loss,Validation Loss,F1 Macro,F1 Weighted
1,0.237858,0.219929,0.929694,0.930489
2,0.154709,0.213485,0.934713,0.935780
3,0.068687,0.240526,0.939822,0.940565
4,0.034313,0.286172,0.942751,0.943535


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=13600, training_loss=0.1407170276562957, metrics={'train_runtime': 1230.6327, 'train_samples_per_second': 353.594, 'train_steps_per_second': 11.051, 'total_flos': 1.4312427604457472e+16, 'train_loss': 0.1407170276562957, 'epoch': 4.0})

In [13]:
# ==========================================
# 7. VALIDATION SET EVALUATION (final chosen model)
# ==========================================
val_metrics = final_trainer.evaluate()
print("==========================================")
print("VALIDATION SET RESULTS (final model)")
print("==========================================")
print(f"Validation Macro-F1:    {val_metrics['eval_f1_macro']:.4f}")
print(f"Validation Weighted-F1: {val_metrics['eval_f1_weighted']:.4f}\n")


Training Loss,Validation Loss,Epoch,F1 Macro,F1 Weighted
0.034313,0.286172,4,0.942751,0.943535


VALIDATION SET RESULTS (final model)
Validation Macro-F1:    0.9428
Validation Weighted-F1: 0.9435



In [14]:
# ==========================================
# 8. FINAL TEST SET EVALUATION (once, on held-out test set)
# ==========================================
test_predictions = final_trainer.predict(test_ds)
y_test_pred_int = np.argmax(test_predictions.predictions, axis=1)
y_test_int = test_df["label"].values

test_f1_macro = f1_score(y_test_int, y_test_pred_int, average="macro")
test_f1_weighted = f1_score(y_test_int, y_test_pred_int, average="weighted")

print("==========================================")
print("FINAL UNSEEN TEST SET RESULTS")
print("==========================================")
print(f"Test Macro-F1:    {test_f1_macro:.4f}")
print(f"Test Weighted-F1: {test_f1_weighted:.4f}\n")

print("Full Classification Report:")
print(classification_report(
    y_test_int, y_test_pred_int, target_names=label_encoder.classes_
))

print("Confusion Matrix (rows=true, cols=predicted):")
cm = confusion_matrix(y_test_int, y_test_pred_int)
cm_df = pd.DataFrame(cm, index=label_encoder.classes_, columns=label_encoder.classes_)
print(cm_df, "\n")

FINAL UNSEEN TEST SET RESULTS
Test Macro-F1:    0.9434
Test Weighted-F1: 0.9441

Full Classification Report:
                 precision    recall  f1-score   support

         اقتصاد       0.93      0.95      0.94      2239
      تكنولوجيا       0.96      0.95      0.95      2328
خواطر واقتباسات       0.93      0.93      0.93      2447
            دين       0.92      0.92      0.92      1957
          رياضه       0.99      0.99      0.99      2327
          سيارت       0.97      0.97      0.97      2214
          سياسه       0.97      0.95      0.96      2437
            صحه       0.90      0.92      0.91      2099
            طهي       0.97      0.96      0.97      1719
           علوم       0.89      0.89      0.89      1990

       accuracy                           0.94     21757
      macro avg       0.94      0.94      0.94     21757
   weighted avg       0.94      0.94      0.94     21757

Confusion Matrix (rows=true, cols=predicted):
                 اقتصاد  تكنولوجيا  خواطر وا

In [15]:
# ==========================================
# 9. SAVING MODEL, TOKENIZER, LABEL ENCODER
# ==========================================
final_trainer.save_model("arabert_final_model")
tokenizer.save_pretrained("arabert_final_model")
joblib.dump(label_encoder, "arabert_label_encoder.joblib")
joblib.dump(best_config, "arabert_best_config.joblib")
print("Saved model, tokenizer, label encoder, and best config to disk.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved model, tokenizer, label encoder, and best config to disk.


In [16]:
predicted_class_names = label_encoder.inverse_transform(y_test_pred_int)
actual_class_names = label_encoder.inverse_transform(y_test_int)

# Create a DataFrame to display test tweets with their predicted classes
results_df = pd.DataFrame({
    'Tweet': test_df[TEXT_COL].values,
    'Actual_Class': actual_class_names,
    'Predicted_Class': predicted_class_names
})

results_df.to_csv('predicted_tweets.csv', index=False)
print("DataFrame exported to 'predicted_tweets.csv'")

DataFrame exported to 'predicted_tweets.csv'
